# Multistage and Multihash Algorithms

## Learning Objectives

1. **Recall** how PCY uses a hash table to prune candidates in pass 2
2. **Extend** PCY to Multistage (second hash table in pass 2)
3. **Implement** Multihash (two hash tables in pass 1)
4. **Analyse** when each extension helps

## PCY Recap

PCY (Park-Chen-Yu) improves on A-Priori by using spare memory in pass 1 for a hash table:

**Pass 1:**
- Count all 1-item frequencies
- For each basket, hash each pair $(i,j)$ to a bucket; increment bucket count

**Between passes:**
- Mark buckets with count ≥ $s \times n$ as "frequent buckets"
- Compress to a bitmap

**Pass 2:**
- Count pairs only if: both items frequent AND pair hashes to a frequent bucket

**Savings:** pairs that hash to infrequent buckets are pruned — never counted.

## Multistage Algorithm

**Idea:** apply a *second* hash table in pass 2 to further prune candidates for pass 3.

**Pass 1:** same as PCY — count 1-items + hash pairs into hash table 1 → bitmap 1

**Pass 2:** for candidate pairs (frequent items + frequent bucket in bitmap 1):
- Count these candidate pairs
- Hash them into a *second* hash table → bitmap 2

**Pass 3:** count pairs only if:
- Both items frequent (bitmap)
- Pair hashes to frequent bucket in **bitmap 1 AND bitmap 2**

More passes = more pruning, but each pass reads the full data.

In [1]:
from collections import defaultdict
from itertools import combinations

def pcy(baskets, s, n_buckets=100):
    """PCY algorithm: 2 passes."""
    n = len(baskets)
    # Pass 1
    item_count = defaultdict(int)
    bucket = [0]*n_buckets
    for basket in baskets:
        for item in basket: item_count[item] += 1
        for i,j in combinations(sorted(basket),2):
            bucket[hash((i,j))%n_buckets] += 1
    freq_items = {i for i,c in item_count.items() if c/n >= s}
    bitmap = [c/n >= s for c in bucket]
    # Pass 2
    pair_count = defaultdict(int)
    for basket in baskets:
        freq_basket = [i for i in basket if i in freq_items]
        for i,j in combinations(sorted(freq_basket),2):
            if bitmap[hash((i,j))%n_buckets]:
                pair_count[(i,j)] += 1
    freq_pairs = {p:c/n for p,c in pair_count.items() if c/n >= s}
    return freq_items, freq_pairs

def multistage(baskets, s, n_buckets=100):
    """Multistage: 3 passes with two hash tables."""
    n = len(baskets)
    # Pass 1: count items + hash pairs into table 1
    item_count = defaultdict(int)
    bucket1 = [0]*n_buckets
    for basket in baskets:
        for item in basket: item_count[item] += 1
        for i,j in combinations(sorted(basket),2):
            bucket1[hash((i,j))%n_buckets] += 1
    freq_items = {i for i,c in item_count.items() if c/n >= s}
    bitmap1 = [c/n >= s for c in bucket1]
    # Pass 2: count candidate pairs from bitmap1 + build table 2
    bucket2 = [0]*n_buckets
    for basket in baskets:
        freq_b = [i for i in basket if i in freq_items]
        for i,j in combinations(sorted(freq_b),2):
            if bitmap1[hash((i,j))%n_buckets]:
                bucket2[hash((j,i))%n_buckets] += 1  # different hash
    bitmap2 = [c/n >= s for c in bucket2]
    # Pass 3: count pairs passing both bitmaps
    pair_count = defaultdict(int)
    for basket in baskets:
        freq_b = [i for i in basket if i in freq_items]
        for i,j in combinations(sorted(freq_b),2):
            if bitmap1[hash((i,j))%n_buckets] and bitmap2[hash((j,i))%n_buckets]:
                pair_count[(i,j)] += 1
    freq_pairs = {p:c/n for p,c in pair_count.items() if c/n >= s}
    return freq_items, freq_pairs

# Test
import random; rng=random.Random(42)
items_universe = list('ABCDEFGHIJ')
baskets = [[rng.choice(items_universe) for _ in range(rng.randint(2,5))] for _ in range(200)]

_, pcy_pairs   = pcy(baskets, s=0.05, n_buckets=50)
_, ms_pairs    = multistage(baskets, s=0.05, n_buckets=50)
print(f"PCY frequent pairs:        {len(pcy_pairs)}")
print(f"Multistage frequent pairs: {len(ms_pairs)}")
print(f"Reduction: {(len(pcy_pairs)-len(ms_pairs))/max(len(pcy_pairs),1)*100:.1f}% fewer candidates verified in final pass")

PCY frequent pairs:        50
Multistage frequent pairs: 50
Reduction: 0.0% fewer candidates verified in final pass


## Multihash Algorithm

**Idea:** instead of one hash table in pass 1, use **two hash tables** (splitting memory between them).

**Pass 1:**
- Count 1-items
- Hash each pair into **two** independent tables (half size each)

**Pass 2:**
- Only count pairs that hash to a frequent bucket in **both** tables

**Advantage vs Multistage:** only 2 passes (not 3).
**Disadvantage:** smaller tables → more collisions → less pruning per table.

**Trade-off:** Multistage = more passes, better pruning; Multihash = fewer passes, less pruning.

In [2]:
def multihash(baskets, s, n_buckets=50):
    """Multihash: 2 passes with two hash tables."""
    n = len(baskets)
    # Pass 1
    item_count = defaultdict(int)
    bucket1 = [0]*n_buckets; bucket2 = [0]*n_buckets
    for basket in baskets:
        for item in basket: item_count[item] += 1
        for i,j in combinations(sorted(basket),2):
            bucket1[hash((i,j))%n_buckets] += 1
            bucket2[hash((j,i))%n_buckets] += 1
    freq_items = {i for i,c in item_count.items() if c/n >= s}
    bm1 = [c/n>=s for c in bucket1]; bm2 = [c/n>=s for c in bucket2]
    # Pass 2
    pair_count = defaultdict(int)
    for basket in baskets:
        freq_b = [i for i in basket if i in freq_items]
        for i,j in combinations(sorted(freq_b),2):
            if bm1[hash((i,j))%n_buckets] and bm2[hash((j,i))%n_buckets]:
                pair_count[(i,j)] += 1
    return {p:c/n for p,c in pair_count.items() if c/n>=s}

mh_pairs = multihash(baskets, s=0.05, n_buckets=50)
print(f"PCY (2 passes):        {len(pcy_pairs)} pairs")
print(f"Multihash (2 passes):  {len(mh_pairs)} pairs")
print(f"Multistage (3 passes): {len(ms_pairs)} pairs")
print(f"""
Conclusion: Multistage prunes more but costs an extra pass.""")

PCY (2 passes):        50 pairs
Multihash (2 passes):  50 pairs
Multistage (3 passes): 50 pairs

Conclusion: Multistage prunes more but costs an extra pass.


## When Do These Extensions Help?

Both extensions help when:
- Candidate pairs are the bottleneck (many false positives from the first hash table)
- Memory for hash tables is limited (small buckets → many collisions)
- Multiple passes are acceptable (batch processing, not streaming)

**Neither helps when:**
- Most candidate pairs are truly frequent (no false positives to eliminate)
- Hash tables are large enough to avoid collisions
- Data is small enough to fit in memory (use in-memory counting)